In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'

train_df = pd.read_csv(f'{BASE_PATH}/train_features.csv')
test_df = pd.read_csv(f'{BASE_PATH}/test_features.csv')

print(train_df.shape, test_df.shape)
print(train_df.columns.tolist())

Mounted at /content/drive
(202583, 36) (50646, 36)
['email_id', 'source_dataset', 'label', 'sender', 'sender_domain', 'datetime', 'day_of_week', 'hour_of_day_utc', 'is_weekend', 'sender_email_count', 'sender_active_days', 'sender_avg_daily_volume', 'interarrival_hours', 'interarrival_ratio', 'is_burst_anomaly', 'url_count', 'body_text', 'domain_age_days', 'tld_phishing_score', 'tld_risk_weight', 'domain_reputation_score', 'has_dmarc_record', 'dmarc_enforced', 'spf_pass', 'dkim_pass', 'ip_is_residential_proxy', 'ip_is_vpn_or_anon', 'ip_is_datacenter', 'ip_reputation_score', 'is_ip_based', 'subdomain_count', 'path_length', 'query_param_count', 'url_length', 'uses_https', 'has_url']


In [ ]:
enron_anom = pd.read_csv(
    f'{BASE_PATH}/enron_emails_cleaned_temporal.csv',
    usecols=['message_id', 'temporal_anomaly', 'temporal_anomaly_reason']
).rename(columns={'message_id': 'email_id'})

nazario_raw = pd.read_csv(
    f'{BASE_PATH}/nazario5_cleaned_temporal.csv',
    usecols=['sender_address', 'utc_datetime', 'temporal_anomaly', 'temporal_anomaly_reason']
)
nazario_raw['utc_datetime'] = pd.to_datetime(nazario_raw['utc_datetime'], errors='coerce')
nazario_raw['sender_norm'] = nazario_raw['sender_address'].astype(str).str.strip().str.lower()
nazario_raw['email_id'] = nazario_raw['sender_norm'] + '_' + nazario_raw['utc_datetime'].astype(str)
nazario_anom = nazario_raw[['email_id', 'temporal_anomaly', 'temporal_anomaly_reason']]

anomalies_all = pd.concat([enron_anom, nazario_anom], ignore_index=True).drop_duplicates(subset='email_id')

train_df = train_df.merge(anomalies_all, on='email_id', how='left')
test_df = test_df.merge(anomalies_all, on='email_id', how='left')

print("Train missing temporal_anomaly:", train_df['temporal_anomaly'].isna().sum())
print("Test missing temporal_anomaly:", test_df['temporal_anomaly'].isna().sum())
print(train_df.groupby('label')['temporal_anomaly'].mean())


Train missing temporal_anomaly: 0
Test missing temporal_anomaly: 0
label
0    0.0
1    0.0
Name: temporal_anomaly, dtype: float64


In [ ]:
train_df.to_csv(f'{BASE_PATH}/train_features.csv', index=False)
test_df.to_csv(f'{BASE_PATH}/test_features.csv', index=False)
print("Patched train/test saved.")

Patched train/test saved.


In [ ]:
train_df = pd.read_csv(f'{BASE_PATH}/train_features.csv')
test_df = pd.read_csv(f'{BASE_PATH}/test_features.csv')

for df in [train_df, test_df]:
    df['datetime'] = pd.to_datetime(df['datetime'])

print(train_df.shape, test_df.shape)
print(train_df[['sender','datetime','interarrival_hours','hour_of_day_utc','day_of_week','is_weekend','is_burst_anomaly','temporal_anomaly']].head())

(202583, 38) (50646, 38)
                    sender                  datetime  interarrival_hours  \
0   don.baughman@enron.com 2002-02-05 17:41:03+00:00            1.771944   
1   chris.foster@enron.com 2001-01-29 14:34:00+00:00           94.950000   
2  georgi.landau@enron.com 2001-10-19 18:55:55+00:00            3.507778   
3   dave.perrino@enron.com 2001-06-01 08:58:00+00:00          597.750000   
4  bmj@michelsonrealty.com 2001-06-06 11:15:00+00:00                 NaN   

   hour_of_day_utc  day_of_week  is_weekend  is_burst_anomaly  \
0             17.0          1.0       False             False   
1             14.0          0.0       False             False   
2             18.0          4.0       False              True   
3              8.0          4.0       False             False   
4             11.0          2.0       False             False   

   temporal_anomaly  
0             False  
1             False  
2             False  
3             False  
4             Fal

In [ ]:
def prep_temporal_features(df):
    df = df.copy()
    df['log_interarrival'] = np.log1p(df['interarrival_hours'].fillna(0).clip(lower=0))
    df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day_utc'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day_utc'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['is_weekend'] = df['is_weekend'].astype(float)
    df['is_burst_anomaly'] = df['is_burst_anomaly'].astype(float)
    df['temporal_anomaly'] = df['temporal_anomaly'].fillna(False).astype(float)
    return df

train_df = prep_temporal_features(train_df)
test_df = prep_temporal_features(test_df)

SEQ_FEATURES = ['log_interarrival','hour_sin','hour_cos','day_sin','day_cos','is_weekend','is_burst_anomaly','temporal_anomaly']
print(train_df[SEQ_FEATURES].describe())

       log_interarrival       hour_sin       hour_cos        day_sin  \
count     202583.000000  202583.000000  202583.000000  202583.000000   
mean           2.180323      -0.323296      -0.243163       0.336294   
std            2.293883       0.626246       0.666462       0.557875   
min            0.000000      -1.000000      -1.000000      -0.974928   
25%            0.110348      -0.866025      -0.866025       0.000000   
50%            1.226060      -0.500000      -0.500000       0.433884   
75%            3.885011       0.258819       0.258819       0.781831   
max           10.699175       1.000000       1.000000       0.974928   

             day_cos     is_weekend  is_burst_anomaly  temporal_anomaly  
count  202583.000000  202583.000000     202583.000000          202583.0  
mean       -0.040458       0.041228          0.164708               0.0  
std         0.757661       0.198817          0.370918               0.0  
min        -0.900969       0.000000          0.000000  

In [ ]:
MAX_SEQ_LEN = 10

def build_sequences(df, seq_features=SEQ_FEATURES, max_len=MAX_SEQ_LEN):
    df = df.copy()
    df['sender'] = df['sender'].fillna('__unknown_sender__')  # prevents groupby from silently dropping these rows
    df = df.sort_values(['sender', 'datetime']).reset_index(drop=True)
    feat_matrix = df[seq_features].values.astype(np.float32)

    n = len(df)
    sequences = np.zeros((n, max_len, len(seq_features)), dtype=np.float32)
    lengths = np.zeros(n, dtype=np.int64)

    for _, idx_group in df.groupby('sender').indices.items():
        idx_group = np.sort(idx_group)
        for pos, row_idx in enumerate(idx_group):
            start = max(0, pos - max_len + 1)
            window_idx = idx_group[start:pos+1]
            L = len(window_idx)
            sequences[row_idx, :L, :] = feat_matrix[window_idx]
            lengths[row_idx] = L

    lengths = np.maximum(lengths, 1)  # safety floor -- should be redundant now, but guards against any other edge case
    return sequences, lengths, df['email_id'].values, df['label'].values

print("Building train sequences...")
train_seq, train_lens, train_ids, train_labels = build_sequences(train_df)
print("Building test sequences...")
test_seq, test_lens, test_ids, test_labels = build_sequences(test_df)

print(train_seq.shape, test_seq.shape)
print("Min/max length:", train_lens.min(), train_lens.max())

Building train sequences...
Building test sequences...
(202583, 10, 8) (50646, 10, 8)
Min/max length: 1 10


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

class SequenceDataset(Dataset):
    def __init__(self, sequences, lengths, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.sequences[idx], self.lengths[idx], self.labels[idx]

train_dataset = SequenceDataset(train_seq, train_lens, train_labels)
test_dataset = SequenceDataset(test_seq, test_lens, test_labels)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(len(train_dataset), len(test_dataset))

Using device: cpu
202583 50646


In [ ]:
class TemporalLSTM(nn.Module):
    def __init__(self, input_size=8, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x, lengths, return_embedding=False):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h_n, c_n) = self.lstm(packed)
        embedding = h_n[-1]  # final layer's hidden state -- this is what we keep as the embedding
        if return_embedding:
            return embedding
        logits = self.classifier(embedding)
        return logits

model = TemporalLSTM().to(device)
print(model)

TemporalLSTM(
  (lstm): LSTM(8, 64, batch_first=True)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=2, bias=True)
  )
)


In [ ]:
label_counts = pd.Series(train_labels).value_counts().sort_index()
total = label_counts.sum()
class_weights = torch.tensor([total / (2*c) for c in label_counts], dtype=torch.float32).to(device)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 5  # LSTM is cheap compared to the transformer, can afford more passes

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for seqs, lens, labels in train_loader:
        seqs, lens, labels = seqs.to(device), lens, labels.to(device)
        optimizer.zero_grad()
        logits = model(seqs, lens)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  avg loss: {total_loss/len(train_loader):.4f}")

Class weights: tensor([ 0.5060, 42.4880])
Epoch 1/5  avg loss: 0.4081
Epoch 2/5  avg loss: 0.3634
Epoch 3/5  avg loss: 0.3491
Epoch 4/5  avg loss: 0.3367
Epoch 5/5  avg loss: 0.3259


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

model.eval()
all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for seqs, lens, labels in test_loader:
        seqs, lens = seqs.to(device), lens
        logits = model(seqs, lens)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=['benign','phishing'], digits=4))
print("ROC-AUC:", roc_auc_score(all_labels, all_probs))
print("PR-AUC:", average_precision_score(all_labels, all_probs))

              precision    recall  f1-score   support

      benign     0.9986    0.8356    0.9099     50050
    phishing     0.0613    0.9010    0.1147       596

    accuracy                         0.8364     50646
   macro avg     0.5299    0.8683    0.5123     50646
weighted avg     0.9876    0.8364    0.9005     50646

ROC-AUC: 0.9318854467679971
PR-AUC: 0.15059360671802302


In [ ]:
model.eval()

def extract_embeddings(dataset, ids, batch_size=512):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_embeds = []
    with torch.no_grad():
        for seqs, lens, labels in loader:
            seqs = seqs.to(device)
            emb = model(seqs, lens, return_embedding=True)
            all_embeds.append(emb.cpu().numpy())
    return np.vstack(all_embeds)

train_temporal_embed = extract_embeddings(train_dataset, train_ids)
test_temporal_embed = extract_embeddings(test_dataset, test_ids)

print(train_temporal_embed.shape, test_temporal_embed.shape)
print("NaNs:", np.isnan(train_temporal_embed).sum(), np.isnan(test_temporal_embed).sum())

(202583, 64) (50646, 64)
NaNs: 0 0


In [ ]:
EMBED_PATH = f'{BASE_PATH}/embeddings'

np.save(f'{EMBED_PATH}/train_temporal_embeddings.npy', train_temporal_embed)
np.save(f'{EMBED_PATH}/test_temporal_embeddings.npy', test_temporal_embed)
pd.DataFrame({'email_id': train_ids}).to_csv(f'{EMBED_PATH}/train_temporal_embeddings_index.csv', index=False)
pd.DataFrame({'email_id': test_ids}).to_csv(f'{EMBED_PATH}/test_temporal_embeddings_index.csv', index=False)

torch.save(model.state_dict(), f'{BASE_PATH}/temporal_lstm_model.pt')

print("Saved temporal LSTM embeddings, index files, and model weights.")

Saved temporal LSTM embeddings, index files, and model weights.
